# MEMC cvičení 8 – Body v kouli a jejich vzdálenost

---

Úloha: vyberme **dva nezávislé body $P, Q$ rovnoměrně v jednotkové kouli** (v rovině kruh, v prostoru koule) a spočtěme **střední hodnotu funkce jejich vzdálenosti** $\rho = \|P - Q\|$:

$$J = \frac{1}{V^2}\int_{B}\!\int_{B} f\bigl(\|P - Q\|\bigr)\,dP\,dQ = E\,f(\rho),\qquad P, Q \sim \text{Rovnoměrně}(B),$$

kde $V$ je objem koule (faktor $1/V^2$ je přesně normování na střední hodnotu). Je to čtyř- (v prostoru šesti-) rozměrný integrál, což je typická úloha pro Monte Carlo.

Cvičení spojuje předchozí látku a ukáže tři věci:

- jak generovat **rovnoměrný bod v kouli**: zamítací metodou (cvičení 6) a **polárními souřadnicemi** (inverzní transformace, cvičení 4)
- že poloměr se **nesmí** losovat rovnoměrně, ale jako $\gamma^{1/d}$
- jak **symetrie** úlohy ušetří polovinu náhodných čísel

In [ ]:
import math
import random
import time

import matplotlib.pyplot as plt

random.seed(1)  # opakovatelné výsledky


def estimate(sampler, n):
    '''Běžné Monte Carlo: průměr n volání generátoru a jeho směrodatná chyba sigma/sqrt(n).'''
    s = s2 = 0.0
    for _ in range(n):
        x = sampler()
        s += x
        s2 += x * x
    mean = s / n
    var = max(s2 / n - mean * mean, 0.0)
    return mean, math.sqrt(var / n)


def check(name, mean, se, exact, k=5):
    '''Vypíše odhad a ověří, že leží do k směrodatných chyb od přesné hodnoty.'''
    print(f"{name}: {mean:.5f} +- {se:.5f}  (přesně {exact:.5f})")
    assert abs(mean - exact) <= k * se, name


def distance(P, Q):
    return math.sqrt(sum((p - q) ** 2 for p, q in zip(P, Q)))

Jak úloha vypadá: v jednotkovém kruhu náhodně volíme **dvojice** bodů $P, Q$ a měříme jejich vzdálenost $\rho$ (úsečka). Střední hodnotu $f(\rho)$ přes všechny dvojice hledáme.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.0))
ax.plot([math.cos(t / 100 * 2 * math.pi) for t in range(101)], [math.sin(t / 100 * 2 * math.pi) for t in range(101)], "k")
for k in range(5):
    pts = []
    for _ in range(2):
        r, phi = math.sqrt(random.random()), 2 * math.pi * random.random()
        pts.append((r * math.cos(phi), r * math.sin(phi)))
    (px, py), (qx, qy) = pts
    ax.plot([px, qx], [py, qy], color=f"C{k}", alpha=0.8)
    ax.plot([px], [py], "o", color=f"C{k}")
    ax.plot([qx], [qy], "s", color=f"C{k}")
    ax.text((px + qx) / 2, (py + qy) / 2, f"{math.hypot(px - qx, py - qy):.2f}", fontsize=9)
ax.set_aspect("equal")
ax.set_title("Dvojice $(P, Q)$ a jejich vzdálenost " + r"$\rho$" + "\n(kroužek = $P$, čtverec = $Q$)")
plt.tight_layout()
plt.show()

## 1. Přesné hodnoty (abychom měli s čím porovnávat)

Pro $f(\rho) = \rho^2$ vyjde výsledek snadno ručně. Pro nezávislé $P, Q$ platí

$$E\|P - Q\|^2 = E\|P\|^2 + E\|Q\|^2 - 2\,E(P\cdot Q) = 2\,E\|P\|^2,$$

protože $E(P\cdot Q) = EP \cdot EQ = 0$ (střed koule je těžiště). Poloměr $r = \|P\|$ má v $d$ rozměrech hustotu $d\,r^{d-1}$ na $[0,1]$ (plocha sféry roste jako $r^{d-1}$), a tedy $E\,r^2 = \dfrac{d}{d+2}$. Výsledek:

$$E\,\rho^2 = \frac{2d}{d+2}\quad\Longrightarrow\quad d = 2:\ 1,\qquad d = 3:\ \tfrac65 = 1.2.$$

Pro $f(\rho) = \rho$ je integrál obtížnější; známe hodnoty

$$d = 2:\ E\rho = \frac{128}{45\pi} \approx 0.9054,\qquad d = 3:\ E\rho = \frac{36}{35} \approx 1.0286.$$

In [ ]:
exact = {
    ("rho", 2): 128 / (45 * math.pi),
    ("rho", 3): 36 / 35,
    ("rho2", 2): 1.0,
    ("rho2", 3): 1.2,
}
for k, v in exact.items():
    print(k, f"{v:.6f}")

## 2. Rovnoměrný bod v kouli: zamítací metoda

Rovnoměrný bod v kouli známe už ze cvičení 6: vezmeme **krychli** $[-1,1]^d$, v ní rovnoměrný bod, a přijmeme ho jen když $\|x\| < 1$. Hustota cílového rozdělení je konstanta na kouli, a tedy přijaté body jsou rovnoměrné v kouli.

Pravděpodobnost přijetí je poměr objemů $V_d / 2^d$. V rovině $\pi/4 \approx 78.5\,\%$, v prostoru $\pi/6 \approx 52.4\,\%$. S dimenzí ale **prudce klesá**:

In [ ]:
def sample_unit_ball(d=2):
    '''Zamítací metoda v krychli [-1, 1]^d.'''
    while True:
        x = [2 * random.random() - 1 for _ in range(d)]
        if sum(c * c for c in x) < 1:
            return x


def objem_koule(d):
    return math.pi ** (d / 2) / math.gamma(d / 2 + 1)


print(f"{'d':>3s} {'V_d':>10s} {'přijetí V_d / 2^d':>20s} {'pokusů na bod':>15s}")
for d in (2, 3, 4, 5, 8, 10, 15, 20):
    p = objem_koule(d) / 2 ** d
    print(f"{d:3d} {objem_koule(d):10.4f} {p:20.2e} {1 / p:15.1f}")

# ověření pokusem pro d = 2, 3, 5
for d in (2, 3, 5):
    pokusy = 0
    prijato = 0
    while prijato < 20_000:
        pokusy += 1
        if sum((2 * random.random() - 1) ** 2 for _ in range(d)) < 1:
            prijato += 1
    print(f"d = {d}: změřená pravděpodobnost přijetí {prijato / pokusy:.4f}, teorie {objem_koule(d) / 2 ** d:.4f}")

Už pro $d = 10$ zamítneme asi $400$ z $401$ pokusů, a pro $d = 20$ je to katastrofa. Pro dvě až tři dimenze je zamítací metoda v pořádku, jinak potřebujeme něco lepšího: **přímo vygenerovat bod v kouli**.

## 3. Polární souřadnice: inverzní transformace

Bod v kouli určíme **poloměrem $r$** a **směrem** (úhlem). Obě části vygenerujeme nezávisle.

### Poloměr: $r = \gamma^{1/d}$, nikoli $\gamma$

Poloměr má hustotu $d\,r^{d-1}$, protože tenká slupka $[r, r+dr]$ má objem úměrný $r^{d-1}\,dr$. Distribuční funkce je $F(r) = r^d$ a její inverze (cvičení 4) dává

$$r = \gamma^{1/d}.$$

Tedy **ne** $r = \gamma$! Rovnoměrně losovaný poloměr by dal příliš mnoho bodů blízko středu (tam je malá plocha, ale hodně bodů). Obrázek to ukáže.

### Směr

- **Rovina:** úhel $\varphi \sim R(0, 2\pi)$, bod $(r\cos\varphi,\ r\sin\varphi)$.
- **Prostor:** Archimedova věta: na sféře je $z = \cos\theta$ **rovnoměrně na $[-1,1]$** (plocha pásu mezi dvěma rovnoběžnými rovinami závisí jen na jejich vzdálenosti) a $\varphi \sim R(0, 2\pi)$. Bod je $r\bigl(\sqrt{1-z^2}\cos\varphi,\ \sqrt{1-z^2}\sin\varphi,\ z\bigr)$.
- **Obecná dimenze:** vektor $d$ nezávislých $N(0,1)$ je rotačně symetrický, takže po normování na délku $1$ dává rovnoměrný směr na sféře (Marsaglia / Muller).

In [ ]:
def sample_unit_ball_polar(d=2):
    '''Polární souřadnice pro d = 2, 3: pevný počet náhodných čísel, žádné zamítání.'''
    r = random.random() ** (1 / d)
    phi = 2 * math.pi * random.random()
    if d == 2:
        return [r * math.cos(phi), r * math.sin(phi)]
    z = 2 * random.random() - 1  # Archimedes: cos(theta) je rovnoměrné na [-1, 1]
    s = math.sqrt(1 - z * z)
    return [r * s * math.cos(phi), r * s * math.sin(phi), r * z]


def sample_unit_ball_gauss(d=2):
    '''Libovolné d: směr z normovaného gaussovského vektoru, poloměr γ^(1/d).'''
    g = [random.gauss(0, 1) for _ in range(d)]
    norma = math.sqrt(sum(c * c for c in g))
    r = random.random() ** (1 / d)
    return [r * c / norma for c in g]


# test: podíl bodů s ||x|| < 1/2 má být (1/2)^d (poměr objemů koulí)
n = 100_000
for d in (2, 3):
    for gen in (sample_unit_ball, sample_unit_ball_polar, sample_unit_ball_gauss):
        m, se = estimate(lambda: float(sum(c * c for c in gen(d)) < 0.25), n)
        check(f"d = {d}, {gen.__name__:22s} P(||x|| < 1/2)", m, se, 0.5 ** d)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.7))
kruh = [(math.cos(t / 100 * 2 * math.pi), math.sin(t / 100 * 2 * math.pi)) for t in range(101)]
M = 1500

# 1) zamítací metoda
prij, zam = [], []
for _ in range(M):
    x = (2 * random.random() - 1, 2 * random.random() - 1)
    (prij if x[0] ** 2 + x[1] ** 2 < 1 else zam).append(x)
axes[0].scatter(*zip(*zam), s=5, c="tab:red", alpha=0.6, label="zamítnuto")
axes[0].scatter(*zip(*prij), s=5, c="tab:green", alpha=0.6, label="přijato")
axes[0].set_title(f"Zamítací metoda: přijato {len(prij) / M:.0%}")
axes[0].legend(loc="lower left", fontsize=8)

# 2) špatně: r = γ
spatne = []
for _ in range(M):
    r, phi = random.random(), 2 * math.pi * random.random()
    spatne.append((r * math.cos(phi), r * math.sin(phi)))
axes[1].scatter(*zip(*spatne), s=5, c="tab:red", alpha=0.7)
axes[1].set_title("ŠPATNĚ: $r = \\gamma$ (shluk ve středu)")

# 3) správně: r = sqrt(γ)
spravne = [sample_unit_ball_polar(2) for _ in range(M)]
axes[2].scatter(*zip(*spravne), s=5, c="tab:green", alpha=0.7)
axes[2].set_title(r"SPRÁVNĚ: $r = \sqrt{\gamma}$ (rovnoměrné)")

for ax in axes:
    ax.plot(*zip(*kruh), "k", lw=1)
    ax.set_aspect("equal")
    ax.set_xlim(-1.05, 1.05)
    ax.set_ylim(-1.05, 1.05)
plt.tight_layout()
plt.show()

Uprostřed je vidět chyba, kterou se vyplatí zapamatovat: stejný počet bodů na menší ploše znamená vyšší hustotu. Správný poloměr $\sqrt{\gamma}$ ($d = 2$) tuto hustotu vyrovná.

## 4. Odhad $J$ pro dva body v kouli

Teď samotná úloha: spočteme průměr $f(\|P-Q\|)$ pro dva nezávislé body, pro obě funkce $f$ a oba generátory.

In [ ]:
def pair_mean(f, d, sampler, n):
    return estimate(lambda: f(distance(sampler(d), sampler(d))), n)


n = 100_000
for nazev, f, klic in (("rho  ", lambda r: r, "rho"), ("rho^2", lambda r: r * r, "rho2")):
    for d in (2, 3):
        for gen in (sample_unit_ball, sample_unit_ball_polar):
            check(f"f = {nazev}  d = {d}  {gen.__name__:22s}", *pair_mean(f, d, gen, n), exact[(klic, d)])

In [ ]:
# rychlost generátorů (jeden bod)
m = 100_000
print(f"{'generátor':26s} {'d':>2s} {'čas na bod':>12s}")
for d in (2, 3):
    for gen in (sample_unit_ball, sample_unit_ball_polar, sample_unit_ball_gauss):
        t0 = time.perf_counter()
        for _ in range(m):
            gen(d)
        print(f"{gen.__name__:26s} {d:2d} {(time.perf_counter() - t0) / m * 1e6:9.2f} us")

Polární souřadnice jsou už v $d = 2, 3$ asi $2$ až $3\times$ rychlejší než zamítání a mají **pevnou cenu** (žádná smyčka). Gaussovský generátor je v nízké dimenzi pomalejší (generuje normální čísla), ale funguje v každé dimenzi, a tam, kde zamítání selhává, je nepostradatelný.

## 5. Symetrie: kolik náhodných čísel opravdu potřebujeme?

Funkce $f(\|P-Q\|)$ závisí jen na **vzdálenosti**, ne na poloze dvojice v prostoru. Otočíme-li obě koule o stejnou rotaci, vzdálenost se nezmění a rozdělení dvojice $(P, Q)$ také ne. Můžeme proto **natočit $P$ na osu** (v prostoru na severní pól, v rovině na osu $x$). Pak zbývá jen **úhel mezi $P$ a $Q$**.

V prostoru: $\mu = \cos\theta$ je rovnoměrné na $[-1,1]$ (Archimedes, jako výše), a podle **kosinové věty**

$$\rho^2 = r_P^2 + r_Q^2 - 2\,r_P\,r_Q\,\mu.$$

V rovině je úhel $\varphi \sim R(0, 2\pi)$ a místo $\mu$ je $\cos\varphi$.

Poloměry $r_P, r_Q$ jsou nezávislé s hustotou $d\,r^{d-1}$, tedy $\gamma^{1/d}$ každý. Počet náhodných čísel: **3 místo 6** (v prostoru), resp. **3 místo 4** (v rovině).

In [ ]:
def rho_symetrie(d):
    '''Vzdálenost dvou rovnoměrných bodů v kouli (d = 2, 3) s využitím symetrie: P je na ose.'''
    rp = random.random() ** (1 / d)
    rq = random.random() ** (1 / d)
    if d == 3:
        mu = 2 * random.random() - 1          # cos(theta) je rovnoměrné na [-1, 1]
    else:
        mu = math.cos(2 * math.pi * random.random())
    rho2 = rp * rp + rq * rq - 2 * rp * rq * mu
    return math.sqrt(max(rho2, 0.0))          # max: ochrana proti -1e-17 z chyb zaokrouhlení


n = 100_000
for nazev, f, klic in (("rho  ", lambda r: r, "rho"), ("rho^2", lambda r: r * r, "rho2")):
    for d in (2, 3):
        check(f"f = {nazev}  d = {d}  symetrie", *estimate(lambda: f(rho_symetrie(d)), n), exact[(klic, d)])

# rychlost
m = 100_000
for d in (2, 3):
    t0 = time.perf_counter()
    for _ in range(m):
        rho_symetrie(d)
    t_sym = (time.perf_counter() - t0) / m
    t0 = time.perf_counter()
    for _ in range(m):
        distance(sample_unit_ball_polar(d), sample_unit_ball_polar(d))
    t_plne = (time.perf_counter() - t0) / m
    print(f"d = {d}: symetrie {t_sym * 1e6:.2f} us, plná dvojice (polární) {t_plne * 1e6:.2f} us  ({t_plne / t_sym:.1f}x pomalejší)")

Pozor: symetrie nezmění **rozptyl** $f(\rho)$ (rozdělení $\rho$ je stejné, jen ho generujeme levněji), takže při stejném $N$ je chyba stejná a vyhráváme na **ceně**. Přesvědčme se, že rozdělení vzdálenosti je opravdu stejné, a podívejme se, jak vypadá.

In [ ]:
N = 200_000
plne3 = [distance(sample_unit_ball_polar(3), sample_unit_ball_polar(3)) for _ in range(N)]
sym3 = [rho_symetrie(3) for _ in range(N)]
plne2 = [distance(sample_unit_ball_polar(2), sample_unit_ball_polar(2)) for _ in range(N)]
sym2 = [rho_symetrie(2) for _ in range(N)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
for ax, plne, sym, d in ((axes[0], plne2, sym2, 2), (axes[1], plne3, sym3, 3)):
    ax.hist(plne, bins=60, range=(0, 2), density=True, alpha=0.5, label="dva nezávislé body")
    ax.hist(sym, bins=60, range=(0, 2), density=True, histtype="step", lw=1.5, color="k", label="symetrie")
    ax.set_title(f"Rozdělení vzdálenosti dvou bodů, $d = {d}$")
    ax.set_xlabel(r"$\rho$")
    ax.legend()
plt.tight_layout()
plt.show()

print(f"d = 3: E rho (plné) = {sum(plne3) / N:.4f}, (symetrie) = {sum(sym3) / N:.4f}, přesně {36 / 35:.4f}")
print(f"d = 2: E rho (plné) = {sum(plne2) / N:.4f}, (symetrie) = {sum(sym2) / N:.4f}, přesně {128 / (45 * math.pi):.4f}")

Histogramy splývají. V rovině je navíc vidět, že vzdálenost $\rho$ nepřesáhne průměr $2$.

## 6. Vyšší dimenze

Gaussovský generátor funguje v libovolné dimenzi. Pro $f(\rho) = \rho^2$ známe $E\rho^2 = \dfrac{2d}{d+2}$ pro **každé** $d$, takže si ověříme třeba $d = 10$, kde by zamítací metoda potřebovala přes $400$ pokusů na bod.

In [ ]:
n = 30_000
for d in (4, 10, 20):
    check(f"d = {d:2d}: E rho^2", *pair_mean(lambda r: r * r, d, sample_unit_ball_gauss, n), 2 * d / (d + 2))

S rostoucí dimenzí se $E\rho^2 \to 2$: téměř všechny body leží blízko **povrchu** koule ($E\,r^2 = d/(d+2) \to 1$) a dva takové body jsou skoro vždy téměř kolmé, tedy $\rho \approx \sqrt 2$. Je to jeden z projevů „prokletí dimenze".

## 7. Alternativa: NumPy

Vektorová verze umí milión párů za zlomek sekundy:

In [ ]:
try:
    import numpy as np

    rng = np.random.default_rng(1)
    N = 1_000_000

    def body_v_kouli(N, d):
        g = rng.standard_normal((N, d))
        g /= np.linalg.norm(g, axis=1, keepdims=True)
        return g * (rng.random(N) ** (1 / d))[:, None]

    for d in (2, 3):
        rho = np.linalg.norm(body_v_kouli(N, d) - body_v_kouli(N, d), axis=1)
        print(f"d = {d}: E rho = {rho.mean():.5f} +- {rho.std(ddof=1) / np.sqrt(N):.5f} (přesně {exact[('rho', d)]:.5f}),"
              f"  E rho^2 = {np.mean(rho ** 2):.5f} (přesně {exact[('rho2', d)]:.5f})")
except ModuleNotFoundError:
    print("NumPy není nainstalované, spusťte:  uv sync")

## Shrnutí

| Pojem | Co si pamatovat |
|-------|-----------------|
| Rovnoměrný bod v kouli, zamítání | bod v krychli $[-1,1]^d$, přijmi když $\|x\|<1$; přijetí $V_d/2^d$ klesá s dimenzí |
| Polární souřadnice | poloměr $r = \gamma^{1/d}$ (inverze $F(r) = r^d$), směr rovnoměrný |
| Směr v rovině / prostoru / obecně | $\varphi \sim R(0,2\pi)$ / Archimedes: $z \sim R(-1,1)$ a $\varphi$ / normovaný gaussovský vektor |
| Častá chyba | $r = \gamma$ místo $\gamma^{1/d}$ → shluk ve středu |
| Symetrie | otoč $P$ na osu; zbývá úhel mezi $P$ a $Q$ a kosinová věta; 3 náhodná čísla místo 6 |
| Přesné hodnoty | $E\rho^2 = 2d/(d+2)$; $E\rho = 128/(45\pi)$ ($d=2$), $36/35$ ($d=3$) |

### Zkuste si

1. **Jednorozměrný případ:** $P, Q \sim R(-1,1)$. Odvoďte $E|P-Q| = \tfrac23$ a $E\rho^2 = \tfrac23$ (z obecného vzorce $2d/(d+2)$ při $d=1$) a ověřte simulací. Platí pro $d = 1$ vzorec pro poloměr $\gamma^{1/d}$?
2. Odhadněte $P(\rho < 1)$ pro $d = 2$ a $d = 3$ třemi způsoby: zamítáním, polárními souřadnicemi a symetrií. Který je nejrychlejší pro stejnou chybu?
3. Odvoďte pravděpodobnost přijetí zamítací metody pro $d = 5$ z $V_5 = \dfrac{8\pi^2}{15}$ a ověřte ji simulací (vyšlo $\approx 16\,\%$).
4. Odhadněte $E\,\rho^4$ pro $d = 2$ (funkce `rho_symetrie`) a pak ji spočtěte ručně z $\rho^2 = \|P\|^2 + \|Q\|^2 - 2P\cdot Q$ (výsledek má být $5/3$).